In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from scipy.stats import kruskal
import scikit_posthocs as sp
import scipy.stats as stats
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split

import sys
import os

# Adds the project root directory to the python path
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))

# Now your import will work perfectly
from src.housing.wrangle import wrangle

In [ ]:
# Wrangling all 5 datasets and concatenating them into one dataframe
df1 = wrangle("/Users/nxcho/Desktop/Data Science/housing_in_Mexico_City/data/mexico_city_real_estate_1.csv")
df2 = wrangle("/Users/nxcho/Desktop/Data Science/housing_in_Mexico_City/data/mexico_city_real_estate_2.csv")
df3 = wrangle("/Users/nxcho/Desktop/Data Science/housing_in_Mexico_City/data/mexico_city_real_estate_3.csv")
df4 = wrangle("/Users/nxcho/Desktop/Data Science/housing_in_Mexico_City/data/mexico_city_real_estate_4.csv")
df5 = wrangle("/Users/nxcho/Desktop/Data Science/housing_in_Mexico_City/data/mexico_city_real_estate_5.csv")

df = pd.concat([df1, df2, df3, df4, df5], ignore_index=True)

df.head()

In [ ]:
# Scatter map plot showing the price of apartments in Mexico City
fig = px.scatter_map(
    df,
    lat = "lat",
    lon = "lon",
    color = "price_aprox_usd",
    center = {"lat": 19.43, "lon": -99.13},
    width = 600,
    height = 600
)

fig.update_layout(map_style = "open-street-map")

fig.show()

In [ ]:
# A histogram to show apartment price distribution
plt.hist(df["price_aprox_usd"])

plt.title("Price Distribution")
plt.xlabel("Price [USD]")
plt.ylabel("Frequency")

plt.show()

In [ ]:
# A scatter plot of price and surface area covered
plt.scatter(df["price_aprox_usd"], df["surface_covered_in_m2"])

plt.xlabel("Price [USD]")
plt.ylabel("Surface covered [m2]")
plt.title("A scatter plot of price and the surface area covered")

plt.show()

In [ ]:
# A bar graph showing the mean price of apartments by borough
mean_price_by_borough = df.groupby("borough")["price_aprox_usd"].mean().sort_values(ascending = True)

mean_price_by_borough.plot(
    kind = "bar",
    xlabel = "Borough",
    ylabel = "Price [USD]"
)

plt.title("A bar graph showing the mean price by borough")

plt.show()

In [ ]:
# Price distribution by borough
order = df.groupby("borough")["price_aprox_usd"].median().sort_values().index

plt.figure(figsize=(8, 8))

sns.boxplot(data=df, y="borough", x="price_aprox_usd", order=order)

plt.xlabel("Price [USD]")
plt.ylabel("Borough")
plt.title("Price Distribution by Borough")

plt.show()

In [ ]:
# Price skewness and variance by borough
print(df.groupby("borough")["price_aprox_usd"].skew().sort_values())
print(df.groupby("borough")["price_aprox_usd"].var().sort_values())

In [ ]:
# Histograms of price distribution by borough
boroughs = sorted(df['borough'].unique())
fig, axes = plt.subplots(len(boroughs), 1, figsize=(8, 4 * len(boroughs)))

for i, borough in enumerate(boroughs):
    subset = df[df['borough'] == borough]
    axes[i].hist(subset['price_aprox_usd'], bins=30, color='skyblue', edgecolor='black')
    axes[i].set_title(f'Price Distribution in {borough}')
    axes[i].set_xlabel('Price [USD]')
    axes[i].set_ylabel('Frequency')

plt.tight_layout()
plt.show()

In [ ]:
# Kruskal-Wallis test across boroughs
groups = [g["price_aprox_usd"].values for _, g in df.groupby("borough")]
stat, p_value = kruskal(*groups)
print("Hstat:", f"{stat:.2e}")
print("P value:", f"{p_value:.2e}")

In [ ]:
# Compute epsilon-squared
n = len(df)
epsilon_sq = stat / (n - 1)
print(f"epsilon^2 = {epsilon_sq:.3f}")

In [ ]:
# Run Dunn's Test
dunn_results = sp.posthoc_dunn(
    df,
    val_col = "price_aprox_usd",
    group_col = "borough",
    p_adjust = "holm"
)

print(dunn_results)

In [ ]:
# Statistically significant pairwise borough price differences (Visualizing Holm-corrected Dunn's test pairwise results)
alpha = 0.05
significant = dunn_results < alpha

plt.figure(figsize=(10, 8))
sns.heatmap(
    significant, 
    cmap=["lightgray", "crimson"],
    cbar=False,
    linewidths=0.5,
    linecolor="white"
)

plt.title("Statistically Significant Pairwise Borough Price Differences\n(Dunn's Test, Holm-corrected, α=0.05)")
plt.show()

In [ ]:
# Correlation between "surface_covered_in_m2" and "price_aprox_usd" by borough
borough_corr = df.groupby("borough")[["surface_covered_in_m2", "price_aprox_usd"]].apply(
    lambda g: g["surface_covered_in_m2"].corr(g["price_aprox_usd"])
).sort_values()

sns.heatmap(
    borough_corr.to_frame(),
    annot = True,
    cmap = "coolwarm",
    vmin = -1,
    vmax = 1,
    fmt = ".2f"
)

plt.title("Surface Area–Price Correlation by Borough")

plt.show()